# Aula 03 — Regressão linear por OLS

Nesta aula, vamos ler a saída da Aula 2 e ajustar um modelo de regressão linear múltipla.

## Objetivos

- consumir a amostra saneada;
- escolher variáveis explicativas;
- ajustar o modelo OLS;
- gerar resíduos e salvar a saída para a Aula 4.

In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Final

import pandas as pd

from servicos.carregamento import resolve_project_root
from servicos.regressao import (
    attach_predictions_and_residuals,
    build_coefficients_table,
    build_model_summary,
    fit_ols_regression,
)

In [ ]:
TARGET_COLUMN: Final[str] = 'preco'
PREFERRED_FEATURES: Final[tuple[str, ...]] = (
    'areaprivativa',
    'vagas',
    'distanciacentrokm',
    'dist_praia',
)


def locate_input_file(project_root: Path) -> Path:
    candidate = project_root / 'data' / 'output' / 'aula_02_amostra_saneada.csv'
    if candidate.exists():
        return candidate
    raise FileNotFoundError('Saída da Aula 2 não encontrada em data/output/.')

## Etapa 1 — Ler a saída da Aula 2

In [ ]:
project_root = resolve_project_root()
input_path = locate_input_file(project_root)
df_model = pd.read_csv(input_path)
print('Base carregada da Aula 2:', df_model.shape)
df_model.head()

## Etapa 2 — Selecionar variáveis

In [ ]:
available_features = [column for column in PREFERRED_FEATURES if column in df_model.columns]
print('Variável dependente:', TARGET_COLUMN)
print('Variáveis explicativas disponíveis:', available_features)

## Etapa 3 — Ajustar o modelo

In [ ]:
model = fit_ols_regression(
    df=df_model,
    target_col=TARGET_COLUMN,
    feature_columns=available_features,
)
summary = build_model_summary(model)
coeff_table = build_coefficients_table(model)
summary

## Etapa 4 — Ler a equação estimada

In [ ]:
def resolve_equation_terms(coeff_table: pd.DataFrame) -> str:
    pieces: list[str] = []
    for _, row in coeff_table.iterrows():
        variable = str(row['variavel'])
        coefficient = float(row['coeficiente'])
        if variable == 'const':
            pieces.append(f'{coefficient:,.2f}')
            continue
        signal = '+' if coefficient >= 0 else '-'
        pieces.append(f'{signal} {abs(coefficient):,.2f}·{variable}')
    return 'Preço = ' + ' '.join(pieces)

print(resolve_equation_terms(coeff_table))
coeff_table

## Etapa 5 — Gerar resíduos e salvar saída

In [ ]:
enriched_df = attach_predictions_and_residuals(df_model, model)
output_dir = project_root / 'data' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'aula_03_amostra_com_residuos.csv'
enriched_df.to_csv(output_path, index=False)
print(output_path)

## Conclusão

A Aula 3 termina com a base modelada e salva. A Aula 4 vai usar essa saída.